In [1]:
import os
os.getcwd()

'C:\\Users\\user\\Downloads\\slm project'

In [2]:
# Uncomment this, when ready to execute

'''
pip install groq --break-system-packages
'''

In [3]:
import os
os.environ["GROQ_API_KEY"] = "replace this set of strings with groq key"

In [4]:
# Checking for model my key has access to 
# Uncomment this, when ready to execute

''' import requests, os

response = requests.get(
    "https://api.groq.com/openai/v1/models",
    headers={"Authorization": f"Bearer {os.environ.get('GROQ_API_KEY')}"}
)
for model in response.json()["data"]:
    print(model["id"])'''

' import requests, os\n\nresponse = requests.get(\n    "https://api.groq.com/openai/v1/models",\n    headers={"Authorization": f"Bearer {os.environ.get(\'GROQ_API_KEY\')}"}\n)\nfor model in response.json()["data"]:\n    print(model["id"])'

In [5]:
from groq import Groq
client = Groq(api_key=os.environ.get("GROQ_API_KEY"))

resp = client.chat.completions.create(
    model="openai/gpt-oss-120b",  # replaces llama-3.3-70b-versatile
    messages=[{"role": "user", "content": "Say hello in one word."}],
    max_tokens=10
)
print(resp.choices[0].message.content)

In [6]:
import os
os.getcwd()

'C:\\Users\\user\\Downloads\\slm project'

In [7]:
# Brtnging in curated corpus
# Only about 20% of curated corpus was used to build this SLM.

import pandas as pd

documents = pd.read_csv('C:\\Users\\user\\Downloads\\slm project\\data\\first_20_percent_combined_df.csv')
documents.head()

,document_id,title,topic,crop,agro_zone,text,origin,source_url,license
0,doc_dis_001,Maize nitrogen deficiency symptoms,crop_diseases,maize,sub_humid,Nitrogen deficiency in maize shows as uniform ...,synthetic,NaN,CC0-1.0
1,doc_dis_002,Common bean rust management,crop_diseases,beans,highland,Bean rust appears as small reddish-brown pustu...,synthetic,NaN,CC0-1.0
2,doc_dis_003,Cassava mosaic disease,crop_diseases,cassava,sub_humid,"Cassava mosaic causes leaf distortion, chlorot...",synthetic,NaN,CC0-1.0
3,doc_pes_001,Fall armyworm in maize,pests,maize,semi_arid,Fall armyworm larvae feed in the whorl leaving...,synthetic,NaN,CC0-1.0
4,doc_pes_002,Stem borer in sorghum,pests,sorghum,semi_arid,Stem borers tunnel inside sorghum stalks causi...,synthetic,NaN,CC0-1.0


In [8]:
documents.tail()

,document_id,title,topic,crop,agro_zone,text,origin,source_url,license
2571,doc_gen_622,Evaluation of 99 Pesticide Residues in Major A...,general,general,highland,There is no information available on pesticide...,cgiar,https://www.mdpi.com/2304-8158/7/11/184/pdf,CC-BY
2572,doc_gen_623,Setting up self-help groups in Nigeria,general,general,sub_humid,A government extension programme in Nigeria is...,cgiar,NaN,NaN
2573,doc_liv_438,"Prevalence of Brucellosis in cattle, sheep, an...",livestock,livestock,sub_humid,A study was designed and carried out between J...,cgiar,https://api.elsevier.com/content/article/pii/S...,CC-BY-NC-ND
2574,doc_gen_624,National Agriculture Investment Plan for Malawi,general,general,sub_humid,Reporting 2021 Policies #199\nNational Agricul...,cgiar,https://cgspace.cgiar.org/rest/bitstreams/8dde...,NaN
2575,doc_dis_489,A report of meloidogyne arenaria parasitizing ...,crop_diseases,general,sub_humid,Extensive root galling observed on plantain (M...,cgiar,NaN,CP


In [9]:
documents.shape

(2576, 9)

In [10]:
"""
generate_train_qa_from_fao_cgiar.py

Builds train_qa specifically from your FAO/CGIAR-sourced documents:
  1. Filters your corpus down to FAO/CGIAR origin only
  2. Cleans institutional-document noise beyond simple whitespace (see CLEANING NOTES)
  3. Generates BOTH question and answer via LLM, grounded strictly in the cleaned text
     (not just extracting the first sentence, unlike the earlier template-based script)
  4. Checkpointed + retried, so a long run over a large corpus survives interruptions

CLEANING NOTES — institutional documents (FAO/CGIAR reports, papers) commonly contain
noise that provides no value for farmer-facing Q&A and can actively pollute generated
answers if left in:
  - Line breaks / carriage returns breaking sentences mid-flow
  - Citation/reference clutter: "(FAO, 2023)", "[12]", "et al."
  - Page furniture: "Page 12 of 45", running headers/footers repeated per page
  - Licensing/boilerplate: "All rights reserved", "This publication is licensed under..."
  - DOIs, ISBNs, URLs embedded mid-paragraph
  - Table-of-contents / figure-caption fragments ("Figure 3.", "Table 2:")
  - Excess whitespace/tabs from PDF extraction
  - Non-ASCII artifacts from PDF-to-text conversion (ligature glitches, stray control chars)

Usage:
    python generate_train_qa_from_fao_cgiar.py
Edit CONFIG below.
"""

'\ngenerate_train_qa_from_fao_cgiar.py\n\nBuilds train_qa specifically from your FAO/CGIAR-sourced documents:\n  1. Filters your corpus down to FAO/CGIAR origin only\n  2. Cleans institutional-document noise beyond simple whitespace (see CLEANING NOTES)\n  3. Generates BOTH question and answer via LLM, grounded strictly in the cleaned text\n     (not just extracting the first sentence, unlike the earlier template-based script)\n  4. Checkpointed + retried, so a long run over a large corpus survives interruptions\n\nCLEANING NOTES — institutional documents (FAO/CGIAR reports, papers) commonly contain\nnoise that provides no value for farmer-facing Q&A and can actively pollute generated\nanswers if left in:\n  - Line breaks / carriage returns breaking sentences mid-flow\n  - Citation/reference clutter: "(FAO, 2023)", "[12]", "et al."\n  - Page furniture: "Page 12 of 45", running headers/footers repeated per page\n  - Licensing/boilerplate: "All rights reserved", "This publication is lice

In [11]:
import re
import os
import json
import time
from pathlib import Path

import pandas as pd

In [12]:

# =========================
# CONFIG
# =========================

CORPUS_PATH = Path("documents.csv")                # your full merged corpus
EXISTING_TRAIN_QA_PATH = Path("train_qa.csv")       # official competition train_qa.csv, for ID offset
OUTPUT_PATH = Path("fao_cgiar_train_qa.csv")
CHECKPOINT_PATH = Path("fao_cgiar_checkpoint.jsonl")

SOURCE_ORIGINS = ["FAO", "CGIAR"]                   # case-insensitive match against the 'origin' column
MIN_DOC_TEXT_CHARS = 120
LLM_PROVIDER = "groq"                                # "groq" (free tier) or "anthropic"
LLM_MODEL = "openai/gpt-oss-20b"               # Groq free-tier model; strong quality, fast
MAX_RETRIES = 3
QUESTIONS_PER_DOC = 1

REQUIRED_DOC_COLUMNS = ["document_id", "title", "topic", "crop", "agro_zone", "text", "origin", "source_url", "license"]


In [13]:
os.getcwd()

'C:\\Users\\user\\Downloads\\slm project'

In [14]:
print(CORPUS_PATH)

documents.csv


In [15]:
print(EXISTING_TRAIN_QA_PATH)

train_qa.csv


In [16]:
from pathlib import Path

print(Path.cwd())

C:\Users\user\Downloads\slm project


## Step 1: Load + filter to FAO/CGIAR

In [17]:

def load_and_filter(path: Path, origins: list[str]) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Corpus not found: {path}")

    df = pd.read_csv(path)
    missing = [c for c in REQUIRED_DOC_COLUMNS if c not in df.columns]
    if missing:
        raise ValueError(f"Corpus missing required columns: {missing}")

    df = df[REQUIRED_DOC_COLUMNS].copy()
    for col in REQUIRED_DOC_COLUMNS:
        df[col] = df[col].fillna("").astype(str)

    origin_lower = df["origin"].str.lower()
    wanted = [o.lower() for o in origins]
    df = df[origin_lower.isin(wanted)].reset_index(drop=True)

    print(f"Filtered to {len(df)} documents with origin in {origins}")
    print(df["origin"].value_counts())
    return df

## Step 2: Deep-clean institutional document noise

In [18]:

CITATION_PATTERN = re.compile(r"\(\s*[A-Z][A-Za-z&.,'\s]{0,40}?,?\s*\d{4}[a-z]?\s*\)")     # (FAO, 2023)
BRACKET_REF_PATTERN = re.compile(r"\[\s*\d+(?:[,\-]\s*\d+)*\s*\]")                          # [12] or [3-5]
PAGE_FURNITURE_PATTERN = re.compile(r"\bPage\s+\d+\s+of\s+\d+\b", re.IGNORECASE)
FIGURE_TABLE_PATTERN = re.compile(r"\b(Figure|Table|Fig\.)\s*\d+[.:]?\s*", re.IGNORECASE)
URL_PATTERN = re.compile(r"https?://\S+")
DOI_PATTERN = re.compile(r"\bdoi:\s*\S+", re.IGNORECASE)
ISBN_PATTERN = re.compile(r"\bISBN[:\s]*[\d\-Xx]+\b")
RIGHTS_BOILERPLATE_KEYWORDS = [
    "all rights reserved", "licensed under", "creative commons", "copyright ©",
    "isbn", "this publication is licensed", "cc by", "attribution-noncommercial",
    "for more information", "for further information", "see figure", "see table",
    "as shown in", "visit our website",
]
CONTROL_CHAR_PATTERN = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f]")
SENTENCE_SPLIT_PATTERN = re.compile(r"(?<=[.!?])\s+")


def deep_clean(text: str) -> str:
    if not text:
        return ""

    text = CONTROL_CHAR_PATTERN.sub(" ", text)
    text = text.replace("\r", " ").replace("\n", " ").replace("\t", " ")

    text = URL_PATTERN.sub(" ", text)
    text = DOI_PATTERN.sub(" ", text)
    text = ISBN_PATTERN.sub(" ", text)
    text = CITATION_PATTERN.sub(" ", text)
    text = BRACKET_REF_PATTERN.sub(" ", text)
    text = PAGE_FURNITURE_PATTERN.sub(" ", text)
    text = FIGURE_TABLE_PATTERN.sub(" ", text)

    # Collapse whitespace left behind before sentence-splitting, so boilerplate
    # sentences (which often span a version number like "3.0 IGO.") split cleanly
    text = re.sub(r"\s+", " ", text).strip()

    # Sentence-level filtering catches boilerplate that a single regex can't safely
    # match (e.g. "licensed under CC BY-NC-SA 3.0 IGO." — the "3.0" period would
    # otherwise truncate an inline regex mid-sentence)
    sentences = SENTENCE_SPLIT_PATTERN.split(text)
    kept = []
    for s in sentences:
        s_stripped = s.strip()
        if not s_stripped:
            continue
        lower = s_stripped.lower()
        if any(kw in lower for kw in RIGHTS_BOILERPLATE_KEYWORDS):
            continue
        # Drop fragments too short to be meaningful (often leftovers from
        # removed citations/URLs, e.g. "For more information visit")
        if len(s_stripped) < 15:
            continue
        kept.append(s_stripped)

    text = " ".join(kept)

    # Final whitespace/punctuation cleanup
    text = re.sub(r"\s+", " ", text).strip()
    text = re.sub(r"(\s[.,;:]){2,}", " ", text)
    text = re.sub(r"\s+([.,;:])", r"\1", text)

    return text.strip()


def clean_corpus(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["text"] = df["text"].apply(deep_clean)
    df["title"] = df["title"].apply(deep_clean)

    before = len(df)
    df = df[df["text"].str.len() >= MIN_DOC_TEXT_CHARS]
    print(f"Dropped {before - len(df)} documents left too short after cleaning (< {MIN_DOC_TEXT_CHARS} chars)")

    before = len(df)
    df = df.drop_duplicates(subset="text").reset_index(drop=True)
    print(f"Dropped {before - len(df)} duplicate documents after cleaning")

    return df



## Step 3: LLM-based Q&A generation (both question AND answer), checkpointed

In [19]:

def build_llm_client():
    """Returns a (provider, client) pair based on LLM_PROVIDER, failing fast and
    clearly if the relevant API key isn't set — better than a cryptic error mid-run."""
    if LLM_PROVIDER == "groq":
        import groq
        api_key = os.environ.get("GROQ_API_KEY")
        if not api_key:
            raise RuntimeError(
                "GROQ_API_KEY not set. Get a free key at https://console.groq.com "
                "and set it: export GROQ_API_KEY=your_key_here"
            )
        return "groq", groq.Groq(api_key=api_key)

    elif LLM_PROVIDER == "anthropic":
        import anthropic
        api_key = os.environ.get("ANTHROPIC_API_KEY")
        if not api_key:
            raise RuntimeError("ANTHROPIC_API_KEY not set in environment.")
        return "anthropic", anthropic.Anthropic(api_key=api_key)

    else:
        raise ValueError(f"Unknown LLM_PROVIDER: {LLM_PROVIDER!r}. Use 'groq' or 'anthropic'.")


def call_llm(provider: str, client, system_prompt: str, user_prompt: str) -> str:
    """Single entry point for both providers, so the calling code doesn't need
    to know which one is active."""
    if provider == "groq":
        resp = client.chat.completions.create(
            model=LLM_MODEL,
            max_tokens=500,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
            ], reasoning_effort = "low"
        )
        return resp.choices[0].message.content.strip()

    elif provider == "anthropic":
        resp = client.messages.create(
            model=LLM_MODEL,
            max_tokens=150,
            system=system_prompt,
            messages=[{"role": "user", "content": user_prompt}],
        )
        return resp.content[0].text.strip()

    raise ValueError(f"Unknown provider: {provider!r}")


def generate_qa_with_llm(documents: pd.DataFrame) -> list[dict]:
    provider, client = build_llm_client()
    print(f"Using LLM provider: {provider} (model: {LLM_MODEL})")

    done_ids = set()
    rows = []
    if CHECKPOINT_PATH.exists():
        with open(CHECKPOINT_PATH, "r", encoding="utf-8") as f:
            for line in f:
                try:
                    row = json.loads(line)
                    rows.append(row)
                    done_ids.add((row["document_id"], row.get("q_index", 0)))
                except json.JSONDecodeError:
                    continue
        print(f"Resuming from checkpoint: {len(done_ids)} question(s) already generated")

    checkpoint_file = open(CHECKPOINT_PATH, "a", encoding="utf-8")

    system_prompt = (
        "You write farmer-facing question-answer pairs strictly grounded in a provided "
        "agricultural/climate document excerpt. Rules:\n"
        "- The question must be something a smallholder farmer would plausibly ask.\n"
        "- The answer must be answerable ONLY from the given excerpt — do not add outside "
        "knowledge, and do not invent facts not present in the text.\n"
        "- The answer must be short, direct, and practical (1-2 sentences, under 50 words).\n"
        "- If the excerpt does not contain enough usable information to form a sensible "
        "farmer question, respond with exactly: SKIP\n"
        "Respond in this exact format, nothing else:\n"
        "QUESTION: <question>\n"
        "ANSWER: <answer>"
    )

    for idx, doc in documents.iterrows():
        for q_index in range(QUESTIONS_PER_DOC):
            if (doc["document_id"], q_index) in done_ids:
                continue

            user_prompt = (
                f"Topic: {doc['topic']} | Crop: {doc['crop']} | Agro-zone: {doc['agro_zone']}\n\n"
                f"Document excerpt:\n{doc['text'][:800]}"
            )

            result_text = None
            for attempt in range(MAX_RETRIES):
                try:
                    result_text = call_llm(provider, client, system_prompt, user_prompt)
                    break
                except Exception as e:
                    wait = 2 ** attempt
                    print(f"  [{doc['document_id']}] attempt {attempt+1}/{MAX_RETRIES} failed: {e} — retrying in {wait}s")
                    time.sleep(wait)

            if result_text is None:
                print(f"  [{doc['document_id']}] failed after {MAX_RETRIES} retries — skipping")
                continue

            if result_text.strip().upper().startswith("SKIP"):
                continue

            q_match = re.search(r"QUESTION:\s*(.+?)(?:\n|$)", result_text, re.IGNORECASE)
            a_match = re.search(r"ANSWER:\s*(.+)", result_text, re.IGNORECASE | re.DOTALL)
            if not q_match or not a_match:
                print(f"  [{doc['document_id']}] unparseable LLM output — skipping: {result_text[:100]}")
                continue

            question = q_match.group(1).strip()
            answer = a_match.group(1).strip()

            if not question or not answer:
                continue

            row = {
                "question": question,
                "topic": doc["topic"],
                "crop": doc["crop"],
                "agro_zone": doc["agro_zone"],
                "document_id": doc["document_id"],
                "reference_answer": answer,
                "q_index": q_index,
            }
            rows.append(row)
            checkpoint_file.write(json.dumps(row) + "\n")
            checkpoint_file.flush()

    checkpoint_file.close()

    # drop the internal q_index bookkeeping column before returning
    for r in rows:
        r.pop("q_index", None)
    return rows


## Step 4: Assign QuestionIds safely + validate + generate train_Q&A

In [ ]:


def finalize(rows: list[dict], existing_qa_path: Path) -> pd.DataFrame:
    df = pd.DataFrame(rows)
    before = len(df)
    df = df.drop_duplicates(subset=["question", "document_id"]).reset_index(drop=True)
    print(f"Dropped {before - len(df)} duplicate question/document pairs")

    start_id = 100000
    if existing_qa_path.exists():
        existing = pd.read_csv(existing_qa_path)
        if "QuestionId" in existing.columns and len(existing) > 0:
            start_id = int(existing["QuestionId"].max()) + 1
        print(f"Offsetting new QuestionIds to start at {start_id}")

    df["QuestionId"] = range(start_id, start_id + len(df))

    # Validation
    assert df["QuestionId"].is_unique
    assert df["question"].str.strip().str.len().gt(0).all()
    assert df["reference_answer"].str.strip().str.len().gt(0).all()
    print(f"Validation passed: {len(df)} rows, {df['document_id'].nunique()} unique documents")

    return df


def main():
    documents = load_and_filter(CORPUS_PATH, SOURCE_ORIGINS)
    documents = clean_corpus(documents)

    if len(documents) == 0:
        raise RuntimeError("No FAO/CGIAR documents survived filtering + cleaning — check origin values and MIN_DOC_TEXT_CHARS.")

    print(f"\nGenerating Q&A via LLM for {len(documents)} documents "
          f"(x{QUESTIONS_PER_DOC} question(s) each)...")
    rows = generate_qa_with_llm(documents)

    if not rows:
        raise RuntimeError("No QA rows generated — check API key, model access, and checkpoint file.")

    result_df = finalize(rows, EXISTING_TRAIN_QA_PATH)
    result_df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8")
    print(f"\nSaved {len(result_df)} rows -> {OUTPUT_PATH}")


if __name__ == "__main__":
    main()


### Clean up corpus and save

In [ ]:
kulal_documents = clean_corpus(documents)

In [ ]:
clean_documents.to_csv("C:\\Users\\user\\Downloads\\slm project\\data\\kulal_documents.csv", index = False)